# 39. The random-direction control

Effective rank falls across a stream: 44.6, 35.0 and 61.2 percent on the three
seeds of `indicator_stream_v1`. Dormant units rise on all three. The direct
probe, measured on those same models in notebook 38, does not fall at all.

So the indicators are saying something, but it may not be what they are read as
saying. **This notebook asks whether they are measuring learning or only
measuring movement.**

## The idea

Over eight tasks the generator travels about 0.12 of its own size. The stream
chose that direction by learning. If a generator pushed the same distance in a
direction chosen by nothing at all shows the same fall, then the fall is a
property of having moved, and the indicator carries no information about what
the movement was for.

Nothing is trained here. A fresh generator is rebuilt from the seed, moved by
hand, and measured. The distances come from the stream's own record, boundary
by boundary, so the control curve lines up with the real curve point for point.

## The two arms

"A random direction" is not one thing, so there are two.

| Arm | What moves | Why it is here |
| --- | --- | --- |
| `global` | one direction drawn evenly over the whole generator, scaled so the whole vector moves by the target distance | the plain reading of "a random direction" |
| `layerwise` | every parameter tensor moves by the target distance of its own size | training does not move every layer by the same absolute amount, so this is the fairer match |

Both arms travel the same total distance. They differ only in how that distance
is spread across the network. If they disagree, how the movement is spread
matters, which is itself worth knowing.

## How to read the answer

- **The control falls like the real run.** Then the fall measures movement, not
  learning, and effective rank says nothing about plasticity in this
  architecture. That is the strongest result this notebook can produce.
- **The control stays flat.** Then the fall is specific to the direction
  learning picked, the indicator is measuring something real, and the puzzle
  moves back to why the direct probe disagrees with it.
- **The control falls part of the way.** Then the honest statement names the
  share, and the indicator is partly a movement meter.

## The check this rests on

Boundary zero of each stream is a fresh generator that the seed rebuilds
exactly. Section 7 rebuilds it and compares its indicators against the numbers
the stream recorded at boundary zero. If they do not match, the rebuild is
wrong and nothing after it means anything, so the cell stops.

## How to run it

Run every cell from the top, in order. Section 2 is the only place to change
anything. A GPU is not required but makes it much faster.

Nothing here trains, so this is far cheaper than a stream run. Section 7 prints
the number of measurements and, after the first one, how long the rest will
take.

## How to restart it if the session stops

Run the notebook again from the top. Every finished measurement is uploaded to
R2, and a measurement is identified by its seed, arm, boundary and repetition
number. Section 6 downloads what is already there and section 7 measures only
what is missing.

Two things to know:

- **Do not change a setting in section 2 between sessions.** Finished rows are
  matched by identity, not by settings, so a changed setting would mix two
  protocols in one file. If you must change something, give `RUN_LABEL` a new
  name first, which starts a separate file.
- **To force a clean rerun**, delete the file named by `RESUME_KEY` in R2.


## 1. Load the code

In [ ]:
from pathlib import Path
import sys, os, json, time, shutil, subprocess, importlib.util

IN_COLAB = "google.colab" in sys.modules
if any(name == "uncle" or name.startswith("uncle.") for name in sys.modules):
    raise RuntimeError("Restart the runtime before loading the code.")

# This notebook runs at the current main, not at a pinned revision. The
# indicators were written after the paired study was frozen, so a pinned
# checkout would not contain them.
if IN_COLAB:
    REPO = Path("/content/path-indicators")
    if not REPO.exists():
        subprocess.run(["git", "clone", "https://github.com/sumitasthana/PATH.git", str(REPO)], check=True)
    subprocess.run(["git", "-C", str(REPO), "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", str(REPO), "checkout", "main"], check=True)
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=True)
else:
    REPO = next((p for p in [Path.cwd(), *Path.cwd().parents]
                 if (p / "uncle/task_partition.json").exists()), None)
    if REPO is None:
        raise FileNotFoundError("Open this notebook from a PATH checkout.")

for package in ("boto3", "tqdm", "matplotlib"):
    if importlib.util.find_spec(package) is None:
        if not IN_COLAB:
            raise ImportError("Install the storage and extras dependencies first.")
        subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", package], check=True)
sys.path.insert(0, str(REPO))

from uncle.storage import R2Store
from uncle.config import Config
from uncle.hypernet import HyperNetwork, build_target
from uncle.trainer import UnCLe
from uncle.plasticity_indicators import (
    dormancy, fixed_image_sets, flatten, indicator_key, ranks, reference_code,
    weight_change)
from uncle.streams import build_tasks

import torch

HEAD = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
print("Running at:", HEAD)
print("PyTorch:", torch.__version__, "| CUDA build:", torch.version.cuda)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU. This will work but slowly; a GPU runtime is much faster.")

## 2. Settings

In [ ]:
# The study whose streams this control is matched to. Nothing here writes to
# it. The study is only read, for its training settings and for the distances
# each seed's stream actually travelled.
STUDY_ID = "indicator_stream_v1"
BUCKET = "uncle-experiments"

ORDER = "order_01"            # tasks 3, 0, 9, 5, 17, 1, 7, 14
SEEDS = [0, 1, 2]             # all three streams that have a finished record

# Names this protocol. Change it before changing anything else below, because
# finished rows are matched by identity and a shared file would silently mix
# two sets of settings.
RUN_LABEL = "control1"

# How the random direction is chosen. See the table at the top. Both arms move
# the generator the same total distance.
ARMS = ["global", "layerwise"]

# How many directions to draw at each distance. A random direction is random,
# so one draw is an anecdote. Three gives a spread to report beside the mean.
# Raise this if the three disagree; every extra repetition is cheap because
# nothing trains.
REPEATS = 3

# Which boundaries to match. The distance for each one is read from the
# stream's own record, so this list only says which points to copy. Boundary 0
# is measured separately and without any movement, as the check that the
# rebuilt generator really is the one the stream started from.
BOUNDARIES = [1, 2, 3, 4, 5, 6, 7, 8]

# The held-out Tiny ImageNet task, the same one the streams and the probe use.
PROBE_TASK = "2"

# The same fixed image sets the stream was measured on. Changing either of
# these makes the control's numbers incomparable with the stream's.
IMAGES_PER_SET = 256
INCLUDE_EUROSAT = True

# The base for the direction seeds. Every draw gets its own seed built from the
# run identity, so a resumed session redraws exactly the same directions.
CONTROL_SEED = 20261103

# How often finished rows are pushed to R2.
SAVE_EVERY = 10

DATA_ROOT = Path("/content/data/tiny-imagenet-200") if IN_COLAB else REPO / "data/tiny-imagenet-200"
EUROSAT_ROOT = Path("/content/eurosat" if IN_COLAB else REPO / "data/eurosat")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Study:", STUDY_ID, "| order:", ORDER, "| seeds:", SEEDS, "| device:", DEVICE)


## 3. Connect to R2

In [ ]:
def secret(name):
    value = os.environ.get(name)
    if not value and IN_COLAB:
        from google.colab import userdata
        value = userdata.get(name)
    if not value:
        raise RuntimeError("Missing secret: " + name)
    return value

store = R2Store.connect(secret("R2_ENDPOINT"), BUCKET,
    secret("R2_ACCESS_KEY_ID"), secret("R2_SECRET_ACCESS_KEY"))
store.progress = True
CACHE = REPO / "outputs/paired_study" / STUDY_ID
(CACHE / "cache").mkdir(parents=True, exist_ok=True)

# Where this control's rows live. A separate folder from the streams, so
# nothing here can be mistaken for a measurement of a trained model.
RESUME_KEY = f"uncle/random_direction/{STUDY_ID}/{ORDER}/{RUN_LABEL}.jsonl"

print("Connected to R2 bucket:", BUCKET)
print("Saving under:", RESUME_KEY)


## 4. Open the study and read the distances to match

In [ ]:
# The study is read, never written. `settings` is the exact configuration the
# streams were trained under, so the generator rebuilt below is built the same
# way theirs was.
STUDY_KEY = "uncle/paired_generalization/" + STUDY_ID + "/study.json"
plan = json.loads(store.download(STUDY_KEY, CACHE / "cache").read_text(encoding="utf-8"))
settings = plan["config"]
ORDER_TASKS = list(plan["orders"][ORDER])
print("Backbone:", settings["backbone"], "| code size:", settings["code_dim"],
      "| chunks:", settings["chunks"])
print("Tasks, in order:", " ".join(ORDER_TASKS))
if PROBE_TASK in ORDER_TASKS:
    raise ValueError("The probe task is in the learning order, so it is not held out.")

# Each stream's own record. Two things are taken from it: how far that stream's
# generator had travelled at each boundary, which is what the control copies,
# and what the indicators read there, which is what the control is compared
# against in section 8.
recorded = {}
for seed in SEEDS:
    key = indicator_key(STUDY_ID, ORDER, seed)
    if store.info(key) is None:
        raise RuntimeError(f"No indicator record for seed {seed} at {key}. "
                           "This control has nothing to match.")
    text = store.download(key, CACHE / "cache").read_text(encoding="utf-8")
    recorded[seed] = {row["t"]: row for row in
                      (json.loads(line) for line in text.splitlines() if line.strip())}
    missing = [t for t in [0] + BOUNDARIES if t not in recorded[seed]]
    if missing:
        raise RuntimeError(f"Seed {seed}'s record is missing boundaries {missing}.")

print()
print("Distances each stream's generator travelled, as a fraction of its own size:")
print("  " + "boundary".ljust(10) + "".join(f"seed {s}".rjust(10) for s in SEEDS))
for t in [0] + BOUNDARIES:
    row = "".join(f"{recorded[s][t]['delta_w_generator']:>10.4f}" for s in SEEDS)
    print(f"  {t:<10}{row}")


## 5. Build the images

In [ ]:
# One call builds every task this session needs. Indexing the dataset is the
# largest cost before the first optimizer step, so the probe task is built here
# with the training tasks rather than in a second pass. The probe task is never
# trained; it only supplies images.
needed = tuple(ORDER_TASKS + [PROBE_TASK])
tasks = build_tasks(root=DATA_ROOT, partition_path=REPO / "uncle/task_partition.json",
                    partition_seed=42, include=needed, download=True)
print("Loaded Tiny ImageNet images for", len(needed), "tasks")

from torch.utils.data import ConcatDataset

sources = {
    # Held-out images of the tasks this stream learns. These say what happens
    # to the representation of data the model is being trained on.
    "in_stream": ConcatDataset([tasks[name]["test"] for name in ORDER_TASKS]),
    # A Tiny ImageNet task the stream never sees. Same dataset, unseen classes.
    "probe": tasks[PROBE_TASK]["test"],
}

if INCLUDE_EUROSAT:
    from torchvision.datasets import EuroSAT
    from torchvision.transforms import ToTensor
    # Satellite images, already 64x64, which is what the backbone expects. A
    # measure that moves on Tiny ImageNet and stays flat here is saying
    # something different from one that moves on both.
    eurosat = EuroSAT(root=str(EUROSAT_ROOT), transform=ToTensor(), download=True)
    assert eurosat[0][0].shape == (3, 64, 64), f"Unexpected shape {eurosat[0][0].shape}"
    sources["eurosat"] = eurosat

# The same pictures at every boundary, in every run, for every seed and order,
# because the draw is seeded inside the library and not here. Only the model
# differs between two measurements.
image_sets = fixed_image_sets(sources, count=IMAGES_PER_SET)
for name, images in image_sets.items():
    print(f"{name:<10} {tuple(images.shape)}")

## 6. Pick up an interrupted run

In [ ]:
# One JSON object per line, so a half-written last line can be dropped without
# losing everything before it.
PARTIAL = CACHE / "diagnostics" / f"random_direction_{RUN_LABEL}.jsonl"
PARTIAL.parent.mkdir(parents=True, exist_ok=True)

results = []
if store.info(RESUME_KEY) is not None:
    saved = store.download(RESUME_KEY, CACHE / "cache")
    for line in saved.read_text(encoding="utf-8").splitlines():
        try:
            results.append(json.loads(line))
        except json.JSONDecodeError:
            print("Skipped one unreadable line at the end of the saved file.")
    print(f"Resuming: {len(results)} measurements already finished.")
else:
    print("Starting fresh: nothing saved in R2 yet.")

PARTIAL.write_text("".join(json.dumps(r) + "\n" for r in results), encoding="utf-8")


def identity(row):
    """Everything that makes one measurement different from another."""
    return (row["seed_label"], row["arm"], row["t"], row["repetition"])


done = {identity(r) for r in results}
planned = len(SEEDS) * (1 + len(ARMS) * len(BOUNDARIES) * REPEATS)
print(f"{planned} measurements in the plan, {planned - len(done)} still to do.")


## 7. Measure the control

This cell does the work. It rebuilds each stream's starting generator, checks
that rebuild against the stream's own boundary-zero numbers, then moves the
generator by hand and measures.

Nothing is trained. Nothing the streams wrote is touched.

In [ ]:
import gc
import random
import time

import numpy as np
from dataclasses import replace
from tqdm.auto import tqdm


def build_fresh(seed):
    """The generator exactly as the stream had it at boundary zero.

    This repeats `run_job`'s own setup: the same Config, the same four random
    seeds set in the same order, then the target network, the hypernetwork and
    the trainer. Nothing is trained and nothing is downloaded, because boundary
    zero is a fresh generator that the seed rebuilds exactly. That claim is
    checked below against the numbers the stream recorded, before anything else
    is measured.

    The trainer is built because the indicator functions need it: it holds the
    clean batch-normalisation statistics every measurement reads through, and
    the device.
    """
    config = replace(Config(**settings), seed=seed, device=DEVICE,
                     requests=tuple(("learn", task) for task in ORDER_TASKS))
    random.seed(config.seed)
    np.random.seed(config.seed)
    torch.manual_seed(config.seed)
    if config.torch_device.type == "cuda":
        torch.cuda.manual_seed_all(config.seed)
    target = build_target(config)
    hypernet = HyperNetwork(target, config)
    trainer = UnCLe(hypernet, config, target, tasks, progress=False)
    return hypernet, target, trainer


def direction_seed(seed, arm, t, repetition):
    """A seed of its own for every draw, built from the run's identity.

    The same row always redraws the same direction, so a resumed session cannot
    quietly change what it is measuring, and two arms at the same boundary do
    not share a direction.
    """
    return (CONTROL_SEED + 100000 * seed + 10000 * ARMS.index(arm)
            + 100 * t + repetition)


@torch.no_grad()
def move_randomly(hypernet, distance, arm, seed):
    """Move the generator `distance` of its own size, in a random direction.

    `global` draws one direction evenly over the whole generator and scales it
    so the whole vector moves by `distance`. The large parameter tensors then
    take almost all of the movement, simply because they hold almost all of the
    entries.

    `layerwise` moves every parameter tensor by `distance` of that tensor's own
    size. Adding the squares back up gives `distance` of the whole vector as
    well, so both arms travel the same total distance and differ only in how it
    is spread. A tensor that starts at exactly zero stays put, and that costs
    nothing: it adds nothing to the whole vector's size either, so the total is
    still exact.

    The distance actually achieved is measured afterwards and recorded, so the
    file says what happened rather than what was intended.
    """
    params = hypernet.generator_parameters()
    stream = torch.Generator(device=params[0].device).manual_seed(seed)
    noise = [torch.randn(p.shape, generator=stream, device=p.device, dtype=p.dtype)
             for p in params]
    if arm == "layerwise":
        for p, n in zip(params, noise):
            size = float(n.norm())
            if size > 0:
                p.add_(n * (distance * float(p.norm()) / size))
    elif arm == "global":
        # Double precision for the two totals. A generator vector has about 135
        # million entries, and a float32 sum over that many terms loses enough
        # digits to shift the distance by more than the effect being measured.
        travelled = float(torch.sqrt(sum((n.double() ** 2).sum() for n in noise)))
        size = float(torch.sqrt(sum((p.double() ** 2).sum() for p in params)))
        scale = distance * size / travelled
        for p, n in zip(params, noise):
            p.add_(n * scale)
    else:
        raise ValueError(f"Unknown arm: {arm}")


def measure(hypernet, target, trainer, start, code):
    """The same indicators the stream recorded, on whatever the generator is now."""
    row = {}
    row.update(weight_change(flatten(hypernet, code), start))
    row.update(ranks(hypernet, target, trainer, image_sets, code))
    row.update(dormancy(hypernet, target, trainer, image_sets, code))
    return row


def save(row):
    results.append(row)
    done.add(identity(row))
    with PARTIAL.open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(row) + "\n")


began = time.perf_counter()
new_rows = 0
worst_move = 0.0
SET_NAMES = list(image_sets)

for seed in SEEDS:
    todo = [(arm, t, rep) for arm in ARMS for t in BOUNDARIES
            for rep in range(REPEATS) if (seed, arm, t, rep) not in done]
    needs_baseline = (seed, "none", 0, 0) not in done
    if not todo and not needs_baseline:
        print(f"seed {seed}: already finished, skipping")
        continue

    print(f"\nseed {seed}: rebuilding the generator the stream started from")
    hypernet, target, trainer = build_fresh(seed)
    code = reference_code(hypernet)
    start = flatten(hypernet, code)
    # The fresh weights, kept on the device, so every draw starts from the same
    # place instead of piling one random move on top of the last.
    fresh = [p.detach().clone() for p in hypernet.generator_parameters()]

    # The check everything else rests on. If the rebuilt generator is not the
    # one the stream started from, every comparison below is meaningless.
    baseline = measure(hypernet, target, trainer, start, code)
    print("  rebuild check, against the numbers the stream recorded at boundary 0:")
    worst = 0.0
    for name in SET_NAMES:
        here = baseline[f"effective_rank_{name}"]
        there = recorded[seed][0][f"effective_rank_{name}"]
        gap = abs(here - there) / there
        worst = max(worst, gap)
        print(f"    effective rank, {name:<10} rebuilt {here:>8.4f}   "
              f"recorded {there:>8.4f}   gap {gap * 100:>5.2f}%")
    if worst > 0.01:
        raise RuntimeError(
            "The rebuilt generator does not match the one the stream started "
            "from, so the control has nothing to compare against. Check that "
            "STUDY_ID, ORDER and the image settings are the stream's own.")
    print("  rebuild matches.")

    if needs_baseline:
        save({**baseline, "seed_label": seed, "arm": "none", "t": 0,
              "repetition": 0, "distance_asked": 0.0, "run_label": RUN_LABEL,
              "device": DEVICE})
        new_rows += 1

    bar = tqdm(todo, desc=f"seed {seed}", unit="measurement")
    for arm, t, rep in bar:
        bar.set_postfix_str(f"{arm} t={t} rep {rep + 1}")
        distance = recorded[seed][t]["delta_w_generator"]
        # Back to the fresh weights first. Without this each draw would land on
        # top of the last one and the distances would add up.
        with torch.no_grad():
            for p, f in zip(hypernet.generator_parameters(), fresh):
                p.copy_(f)
        move_randomly(hypernet, distance, arm, direction_seed(seed, arm, t, rep))
        row = measure(hypernet, target, trainer, start, code)
        # The check on the move itself. The whole comparison rests on the
        # control travelling the same distance the stream did, so if the move
        # missed, stop rather than record a number that looks like an answer.
        # Both arms are exact on paper, so the slack here is for float32
        # rounding over 135 million entries, not for the formula.
        off = abs(row["delta_w_generator"] - distance) / distance
        worst_move = max(worst_move, off)
        if off > 0.02:
            raise RuntimeError(
                f"Asked for a move of {distance:.4f} and got "
                f"{row['delta_w_generator']:.4f}, which is {off * 100:.1f} "
                f"percent out. Arm {arm}, seed {seed}, boundary {t}.")
        save({**row, "seed_label": seed, "arm": arm, "t": t, "repetition": rep,
              "distance_asked": distance, "run_label": RUN_LABEL, "device": DEVICE})
        new_rows += 1
        if new_rows % SAVE_EVERY == 0:
            store.upload(PARTIAL, RESUME_KEY, overwrite=True)
    bar.close()

    del hypernet, target, trainer, fresh
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    store.upload(PARTIAL, RESUME_KEY, overwrite=True)

store.upload(PARTIAL, RESUME_KEY, overwrite=True)
print(f"\n{len(results)} measurements in total, {new_rows} in this session "
      f"({(time.perf_counter() - began) / 60:.1f} minutes).")
print(f"Every move landed within {worst_move * 100:.2f} percent of the "
      "distance it was asked for.")
missing = [(s, a, t, r) for s in SEEDS for a in ARMS for t in BOUNDARIES
           for r in range(REPEATS) if (s, a, t, r) not in done]
if missing:
    print(f"{len(missing)} measurements still to do. Run the notebook again "
          "from the top; everything already finished is skipped.")
else:
    print("Nothing left to run.")


## 8. Read the result

The table puts the stream's own curve beside the control's, boundary by
boundary, at matched distances.

In [ ]:
import statistics as st

SET = "probe"   # the held-out task's images, the set the direct probe trains on


def rows_at(seed, arm, t):
    return [r for r in results if r["seed_label"] == seed
            and r["arm"] == arm and r["t"] == t]


for seed in SEEDS:
    base = recorded[seed][0][f"effective_rank_{SET}"]
    print(f"\n{'=' * 78}\nSeed {seed}, effective rank on the `{SET}` images. "
          f"Fresh generator: {base:.4f}")
    header = f"  {'t':>2} {'distance':>9} {'stream':>9}"
    for arm in ARMS:
        header += f" {arm:>10} {'spread':>7}"
    print(header)
    for t in BOUNDARIES:
        real = recorded[seed][t][f"effective_rank_{SET}"]
        line = (f"  {t:>2} {recorded[seed][t]['delta_w_generator']:>9.4f} "
                f"{real:>9.4f}")
        for arm in ARMS:
            got = [r[f"effective_rank_{SET}"] for r in rows_at(seed, arm, t)]
            if got:
                spread = max(got) - min(got) if len(got) > 1 else 0.0
                line += f" {st.mean(got):>10.4f} {spread:>7.4f}"
            else:
                line += f" {'':>10} {'':>7}"
        print(line)

    # The question in one line per arm: of the fall the stream showed, how much
    # does moving at random reproduce?
    end = recorded[seed][BOUNDARIES[-1]]
    real_fall = base - end[f"effective_rank_{SET}"]
    print(f"\n  The stream's fall by boundary {BOUNDARIES[-1]}: "
          f"{real_fall:.4f} ({real_fall / base * 100:.1f} percent of where it started)")
    for arm in ARMS:
        got = [r[f"effective_rank_{SET}"] for r in rows_at(seed, arm, BOUNDARIES[-1])]
        if not got:
            continue
        fall = base - st.mean(got)
        print(f"  Random movement, {arm:<10} fall {fall:>7.4f} "
              f"({fall / base * 100:>5.1f} percent), which is "
              f"{fall / real_fall * 100:>5.1f} percent of the stream's fall")

print(f"\n{'=' * 78}\nDormant units at 0.01 on the `{SET}` images")
for seed in SEEDS:
    base = recorded[seed][0][f"dormant_0.01_{SET}"]
    end = recorded[seed][BOUNDARIES[-1]][f"dormant_0.01_{SET}"]
    print(f"\n  Seed {seed}: fresh {base:.4f}, stream ends {end:.4f}, "
          f"rise {end - base:+.4f}")
    for arm in ARMS:
        got = [r[f"dormant_0.01_{SET}"] for r in rows_at(seed, arm, BOUNDARIES[-1])]
        if got:
            print(f"    random, {arm:<10} {st.mean(got):.4f}, "
                  f"rise {st.mean(got) - base:+.4f}")

print()
print("Reading it: a control that falls as far as the stream means the "
      "indicator is measuring movement, not learning. A control that stays "
      "near the fresh value means the fall belongs to the direction learning "
      "chose. Anything between the two is partial, and the share above is the "
      "honest way to report it.")
